In [1]:
import pandas as pd
import numpy as np
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler

In [ ]:
features = ['aqi', 'so2', 'co', 'o3', 'pm10', 'pm2.5', 'no2', 'nox', 'no']

In [ ]:
# 2. Load the trained model and dataset for scaling context
model = keras.models.load_model('model_trained.keras')
df = pd.read_csv('cleaned_dataset_taiwan_2months.csv')

In [ ]:
dataset = df[features].values
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(dataset)

MinMaxScaler()

In [ ]:
def predict_air_quality(input_values):
    """
    input_values: list of 9 floats corresponding to features
    """
    # Create DataFrame and scale the input
    sample_df = pd.DataFrame([input_values], columns=features)
    scaled_sample = scaler.transform(sample_df)
    
    # Reshape for LSTM: (1, window_size, n_features)
    # Replicate the point 100 times to match the model's expected window
    X_input = np.repeat(scaled_sample[np.newaxis, :, :], 100, axis=1)
    # Generate scaled prediction
    prediction_scaled = model.predict(X_input, verbose=0)
    
    # Inverse scale to get actual AQI value
    dummy = np.zeros((1, 9))
    dummy[0, 0] = prediction_scaled[0, 0]
    predicted_aqi = scaler.inverse_transform(dummy)[0, 0]
    
    # Categorize based on standards
    bins = [0, 50, 100, 150, 200]
    labels = ["Good", "Moderate", "Unhealthy (Sensitive)", "Unhealthy"]
    idx = np.digitize(predicted_aqi, bins) - 1
    idx = np.clip(idx, 0, len(labels) - 1)
    
    return predicted_aqi, labels[idx]

In [ ]:
sample_point = [50.0, 0.9, 0.17, 35.0, 18.0, 17.0, 12.3, 12.6, 0.3]
aqi_val, status = predict_air_quality(sample_point)

c:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [20]:
print(f"Predicted AQI: {aqi_val:.2f}")
print(f"Air Quality Status: {status}")

Predicted AQI: 43.18
Air Quality Status: Good
